In [2]:
# !pip install pygame

In [3]:
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk, scrolledtext, filedialog, messagebox, PhotoImage
import tkinter.font as tkFont
import threading
# import multiprocessing
import wave
import os
import librosa
import time
import joblib
import warnings
from IPython.display import clear_output
warnings.filterwarnings("ignore")

** Can choose between cuda or cpu option in the device option in perform_initialization **

In [4]:
def perform_initialization():
    #Initialize the whisper model
    #Setup faster-whisper parameters
    model_size = "medium.en" #Choosing model: can choose from: small.en, medium.en, base.en
    device = "cuda" #Choosing between GPU (cuda) or CPU (cpu)
    compute_type = "float16" #float16 mainly for gpu (cuda), int8 for cpu"
    #Initiate whisper model
    whisper_model = WhisperModel(model_size, device = device, compute_type = compute_type)

    #Obtain the dictionary of labels and their encoded values for prediction
    load_labelencoder = joblib.load("./SavedModel/LabelEncoder.bin")
    classes = dict(zip(load_labelencoder.transform(load_labelencoder.classes_), load_labelencoder.classes_,))
    return (whisper_model, classes)

In [5]:
def extract_features(_FilePath):
  mfcc_features = 100
  x, sample_rate = librosa.load(_FilePath, res_type = "kaiser_fast")
  mfcc = np.mean(librosa.feature.mfcc(y = x, sr = sample_rate, n_mfcc = mfcc_features).T, axis = 0)
  return mfcc

In [6]:
def update_emotion_count(_predicted_emotion, _emotion_count):
    if _predicted_emotion in _emotion_count:
        _emotion_count[_predicted_emotion] += 1
    return _emotion_count

# Process Microphone input

In [7]:
from faster_whisper import WhisperModel
from keras.models import load_model
#Speech to text using faster-whisper
def transcribe_audio_live(_self, _file_name, _whisper_model, _emotion_count):
    segments, _ = _whisper_model.transcribe(_file_name,
                                           vad_filter = True,
                                           beam_size = 8)
    for segment in segments:
        input_features = extract_features(_file_name)
        input_features_reshaped = input_features.reshape(1, -1)
        #Scale the predicted data with the saved scaler
        load_scaler = joblib.load("./SavedModel/Scaler.bin")
        mfcc_test_scaled = load_scaler.transform(input_features_reshaped)
        mfcc_test_reshaped = np.reshape(mfcc_test_scaled, (1, 1, input_features_reshaped.shape[1]))
        
        #Load the saved lstm model
        load_lstm = load_model("./SavedModel/LSTM_model.keras")
        predict = load_lstm.predict(mfcc_test_reshaped)
        predict = np.argmax(predict, axis = 1)
        predict = classes[int(predict)]
        #Update the emotion count
        emotion_count = update_emotion_count(predict, _emotion_count)
        #Produce the text output and append it to the scrolled text window
        text_output = f"\n{segment.text} ( {predict} )\n \n"
        append_text(text_output, _self.transcription_window, False, _, _, _self)
        if sum(emotion_count.values()) != 0:
            update_emotion_percentage(_self, emotion_count)

In [8]:
#Record audio with microphone input
import sounddevice as sd
from scipy.io.wavfile import write

def audio_record(file_name, self):
    chunk_duration = 4 #in seconds
    channels = 1
    fs = 16000 #or 44100
    #Initiate recording
    try:
        audio_recording = sd.rec(chunk_duration * fs, samplerate = fs, channels = channels)
        sd.wait()
        #Create wav file from recorded audio
        write(file_name, fs, audio_recording)
    except Exception  as e:
        #Indicate to user that microphone failed to record
        messagebox.showinfo(message = "Microphone failed to record... Please activate your microphone.")
        self.is_recording.set(False)
        #Hide the stop recording button and show the start recording button
        self.stop_record_button.place_forget()
        self.record_button.place(relx = 0.5, rely = 0.9, relwidth = 0.15, relheight = 0.15, anchor = 'center')
        #Re-enable the radiobutton settings
        self.option_microphone_radiobutton.configure(state = "normal")
        self.option_import_radiobutton.configure(state = "normal")
        print("Stopped Recording")
        return

In [9]:
import time
def perform_live_recording(self):
    #Create a dictionary to tally the number of each emotions
    emotion_count = dict.fromkeys(classes.values(), 0)
    while self.is_recording.get():
        #Get the recorded chunk audio file from the microphone input
        threading.Thread(target = audio_record("chunk_audio.wav", self), daemon = True).start()
        transcribe_audio_live(self, "chunk_audio.wav", whisper_model, emotion_count)

# Process Audio File import input

In [12]:
from scipy.io import wavfile
import noisereduce as nr
from pydub import AudioSegment
def preprocess_audio_input(_audio_file):
    #Pre-process the audio file input
    # load the data of the original audio file
    audio_data, sample_rate = librosa.load(_audio_file)
    
    #Reshape before inputing in noise reduction function
    if (len(audio_data) % 2 != 0): #Adjust odd length of data into even before reshaping
      audio_data = np.delete(audio_data, - 1)
    audio_data_reshape = np.reshape(audio_data, (2, -1))
    
    #Perform noise reduction
    audio_noised_reduced = nr.reduce_noise(y = audio_data_reshape, sr = sample_rate)
    #Save the reduced noise audio file into a new wav file
    wavfile.write("./NoiseRemoved.wav", sample_rate, audio_noised_reduced.reshape(audio_data.shape))

In [13]:
from faster_whisper import WhisperModel
from pydub.playback import play
import datetime
def produce_transcribed_output(self, transcription_window):
    #Re-load the reduced noise file
    audio_file = AudioSegment.from_file("./NoiseRemoved.wav")
    #Perform Transcription with whisper
    segments, info = whisper_model.transcribe("./NoiseRemoved.wav", beam_size=5)
    #Create a dictionary to tally the number of each emotions
    emotion_count = dict.fromkeys(classes.values(), 0)
    #Create a dictionary to store the timestamp along with the start and end time
    timestamp_dict = {}
    tag_id = 0
    for segment in segments:
        #Obtain the chunk to process by getting the start and end time of the segmented transcript
        segment_start_time = segment.start * 1000 #Get the start time in milliseconds
        segment_end_time = segment.end * 1000 #Get the end time in milliseconds
        segment_audio = audio_file[segment_start_time:segment_end_time]
        segment_audio.export("./chunk_audio.wav", format = "wav")
        
        #Extract the features of the current chunk
        input_features = extract_features("./chunk_audio.wav")
        input_features_reshaped = input_features.reshape(1, -1)
        #Scale the predicted data with the saved scaler
        load_scaler = joblib.load("./SavedModel/Scaler.bin")
        mfcc_test_scaled = load_scaler.transform(input_features_reshaped)
        mfcc_test_reshaped = np.reshape(mfcc_test_scaled, (1, 1, input_features_reshaped.shape[1]))
        
        #Load the saved lstm model and perform prediction
        load_lstm = load_model("./SavedModel/LSTM_model.keras")
        predict = load_lstm.predict(mfcc_test_reshaped)
        predict = np.argmax(predict, axis = 1)
        predict = classes[int(predict)]
        #Update the emotion count
        emotion_count = update_emotion_count(predict, emotion_count)
        
        #Append the text output for the timestamp that can be clickable
        timestamp = datetime.timedelta(seconds = round(segment.start))
        timestamp_text_output = f"[{timestamp}]"
        timestamp_unique_id = f"tag{tag_id}"#Create a unique tag ID to help identify the timestamp when clicked
        #Add the timestamp (key) and segment start+end time (values) in the dictionary
        timestamp_entry = {timestamp_text_output: [segment.start, segment.end]}
        append_text(timestamp_text_output, transcription_window, True, timestamp_unique_id, timestamp_entry, self)

        #Append the text transcript with the predicted emotion
        transcription_text_output = f" {segment.text} ( {predict} )\n \n"
        append_text(transcription_text_output, transcription_window, False, timestamp_unique_id, timestamp_entry, self)
        update_emotion_percentage(self, emotion_count)
        tag_id += 1
    transcription_window.config(state = "disabled")

In [14]:
def import_audio_file(self):
    RootImport = tk.Tk()
    RootImport.withdraw()
    RootImport.call('wm', 'attributes', '.', '-topmost', True)
    #Launch gui to input file
    audio_file = filedialog.askopenfilename(filetypes = [("Audio files", "*.mp3 *.wav" )]) #Only allow .wav and.mp3 as input
    self.audio_file = audio_file
    #Check if a file was successfully loaded
    try:
        if audio_file:
            #Clear the previous text output
            self.transcription_window.config(state = "normal")
            self.transcription_window.delete('1.0', tk.END)
            self.transcription_window.config(state = "disabled")
            RootImport.destroy()

            #Disable the import button to prevent unwanted errors while processing 
            self.import_button["state"] = tk.DISABLED
            #Disable the radiobutton settings while processing input file to prevent unwanted errors
            self.option_microphone_radiobutton.configure(state = "disabled")
            self.option_import_radiobutton.configure(state = "disabled")
            reset_emotion_percentage(self)
            
            #Pre-process the audio file input
            preprocess_audio_input(audio_file)
            #Perform the entire text-transcript-prediciton output
            produce_transcribed_output(self, self.transcription_window)
        
            #Re-enable the radiobutton after process is complete
            self.option_microphone_radiobutton.configure(state = "normal")
            self.option_import_radiobutton.configure(state = "normal")
            self.import_button["state"] = tk.NORMAL
        else:
            raise SystemExit("NoFile")
    except SystemExit:
        #Indicate to user that file was not loaded via popup messagebox
        messagebox.showinfo(message = "No File was selected... Please try again!")
        RootImport.destroy()
        return
    return

# Tkinter functions

In [15]:
from pygame import mixer
def play_segmented_audio(_event, _tag_id, _timestamp_dict, _self):
    #Get the text of the timestamp
    index = _event.widget.index(f"@{_event.x},{_event.y}")
    tags = _event.widget.tag_names(index)
    for tag in tags:
        if tag.startswith(_tag_id):
            #Get the specific clicked linked tag
            ranges = _event.widget.tag_ranges(tag)
            #Obtain the text of this clicked linked tag
            timestamp_text = _event.widget.get(ranges[0], ranges[1])

            #Match the timestamp text with the key of the dictionary to obtain the starting timestamp of audio file to be played
            start_time, end_time = _timestamp_dict[timestamp_text]
            #Calculate the fadeout time
            fadeout_time = round((end_time - start_time) * 1000) + 500 #Add some time offset to ensure that the entire audio segment is played
            #Play the segmented clip from the given start and fade out time
            audio_file = _self.audio_file
            mixer.init()
            if not mixer.music.get_busy():
                mixer.music.stop() #Stop any currently running segmented audio file
                mixer.music.unload() #Unload any currently running segmented audio file
                mixer.music.load(audio_file)
                mixer.music.play(start = start_time)
                mixer.music.fadeout(fadeout_time)
                mixer.music.unload()
            break

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [16]:
def append_text(_text, _transcription_window, _isclickable, _tag_id, _timestamp_dict, self):
    _transcription_window.config(state = "normal")
    #If the text is supposed to be clickible then add the link tag
    if _isclickable:
        _transcription_window.insert(tk.END, _text, _tag_id)
        _transcription_window.tag_config(_tag_id, foreground = "blue", underline = True)
        _transcription_window.tag_bind(_tag_id, "<Button-1>", lambda e:threading.Thread(target = play_segmented_audio, 
                                                                                        args = (e, _tag_id, _timestamp_dict, self,),
                                                                                        daemon = True).start())
    else:
        _transcription_window.insert(tk.END, _text)
    _transcription_window.see(tk.END)
    _transcription_window.config(state = "disabled")
    #transcription_window.update()

In [17]:
def update_emotion_percentage(self, _emotion_count):
    #Get the total number of emotions
    total = sum(_emotion_count.values())
    #Update the labels with the percentage of each emotion
    self.label_happy.config(text = f"Happy {_emotion_count["Happy"] / total * 100:.0f}%")
    self.label_surprise.config(text = f"Surprised {_emotion_count["Surprised"] / total * 100:.0f}%")
    self.label_neutral.config(text = f"Neutral {_emotion_count["Neutral"] / total * 100:.0f}%")
    self.label_disgust.config(text = f"Disgust {_emotion_count["Disgust"] / total * 100:.0f}%")
    self.label_sad.config(text = f"Sad {_emotion_count["Sad"] / total * 100:.0f}%")
    self.label_fearful.config(text = f"Fearful {_emotion_count["Fearful"] / total * 100:.0f}%")
    self.label_angry.config(text = f"Angry {_emotion_count["Angry"] / total * 100:.0f}%")

In [18]:
def reset_emotion_percentage(self):
    self.label_happy.config(text = f"Happy 0%")
    self.label_surprise.config(text = f"Surprised 0%")
    self.label_neutral.config(text = f"Neutral 0%")
    self.label_disgust.config(text = f"Disgust 0%")
    self.label_sad.config(text = f"Sad 0%")
    self.label_fearful.config(text = f"Fearful 0%")
    self.label_angry.config(text = f"Angry 0%")

In [19]:
def close_application(_is_recording):
    if messagebox.askokcancel("Quit", "Do you want to quit the application?"):
        _is_recording.set(False) #Ensures that recording is stopped before exiting application
        #Removes the recorded files
        if os.path.exists("./chunk_audio.wav"):
            os.remove("chunk_audio.wav")
        if os.path.exists("./NoiseRemoved.wav"):
            os.remove("NoiseRemoved.wav")
        GUI.destroy()

# GUI Launch

In [ ]:
#Create GUI window
import time
class GuiComponents():
    def __init__(self, master, _whisper_model, _classes):
        #Initialize variables
        self.master = master
        self.is_recording = tk.BooleanVar(value = False)
        self.input_option = tk.IntVar()
        self.audio_file = ""
        #Icons
        self.record_icon = PhotoImage(file = './GUI Icons/MicrophoneRecord.png')
        self.stop_icon = PhotoImage(file = './GUI Icons/StopIcon.png')
        self.import_icon = PhotoImage(file = './GUI Icons/ImportFile.png')
        self.microphone_icon = PhotoImage(file = './GUI Icons/Microphone.png')
        self.file_icon = PhotoImage(file = './GUI Icons/AudioFile.png')
        ################################## CREATING GUI COMPONENTS #####################################
        #Create Title
        title_font = tkFont.Font(family = "Impact", size = 22, weight = "normal")
        self.title = ttk.Label(GUI,
                              text = "Speech To Emotion (STE)",
                              font = title_font,
                              foreground = "black",
                              background = "light steel blue",)
        self.title.place(relx = 0.55, rely = 0.05, relwidth = 0.3, relheight = 0.1, anchor = 'center')
                    
        #Create a Scrolled text window to show the text output
        self.transcription_window = scrolledtext.ScrolledText(GUI,
                                                              wrap = tk.WORD,
                                                              width = 110,
                                                              height = 50,
                                                              font = ("Verdana", 15),
                                                              bg = "floral white",
                                                             )
        self.transcription_window.place(relx = 0.15, rely = 0.075, relwidth = 0.75, relheight = 0.75)


        ############################### RADIO BUTTON SELECTIONS ##############################
        #Create a radiobutton to allow users to select between input options
        self.frame_radiobuttons = tk.Frame(GUI, bg = "floral white", relief = tk.GROOVE, borderwidth = 2)
        self.frame_radiobuttons.place(relx = 0.91, rely = 0.25, relwidth = 0.1, relheight = 0.3)
        self.option_label = tk.Label(GUI, bg = "gainsboro", font = ("Verdana 10 bold"), relief = tk.RIDGE, borderwidth = 2,
                                     text = "Audio Input: ")
        self.option_label.place(relx = 0.91, rely = 0.2, relwidth = 0.1, relheight = 0.05)
                                     
        #1.) Radiobar for microphone input
        self.option_microphone_radiobutton = tk.Radiobutton(self.frame_radiobuttons, 
                                                           image = self.microphone_icon,
                                                           bg = "floral white",
                                                           variable = self.input_option,
                                                           value = 0,
                                                           command = self.show_microphone_input_buttons
                                                          )
        self.option_microphone_radiobutton.place(relx = 0.5, rely = 0.25, relwidth = 0.8, relheight = 0.3, anchor = 'center') 
        #2.) Radiobar for audio file import
        self.option_import_radiobutton = tk.Radiobutton(self.frame_radiobuttons, 
                                                       image = self.file_icon,
                                                       bg = "floral white",
                                                       variable = self.input_option,
                                                       value = 1,
                                                       command = self.show_import_audio_file
                                                       )
        self.option_import_radiobutton.place(relx = 0.5, rely = 0.7, relwidth = 0.8, relheight = 0.3, anchor = 'center') 
        #set the default input option on startup
        self.input_option.set(0)

        ############################## Text Labels ###############################
        #Create a text label to show the emotion counts
        self.frame_emotion_count = tk.Frame(GUI, bg = "floral white", relief = tk.GROOVE, borderwidth = 2)
        self.frame_emotion_count.place(relx = 0.0, rely = 0.125, relwidth = 0.14, relheight = 0.7)
        self.option_label = tk.Label(GUI, bg = "gainsboro", font = ("Verdana 10 bold"), relief = tk.RIDGE, borderwidth = 2,
                                     text = "Emotions Detected ")
        self.option_label.place(relx = 0.0, rely = 0.075, relwidth = 0.14, relheight = 0.05)
        #Create labels
        self.label_happy = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Happy 0%", bg = "floral white")
        self.label_happy.place(relx = 0.5, rely = 0.05, relwidth = 1.0, relheight = 0.15, anchor = 'center')
        self.label_surprise = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Surprised 0%", bg = "floral white")
        self.label_surprise.place(relx = 0.5, rely = 0.2, relwidth = 1.0, relheight = 0.15, anchor = 'center')
        self.label_neutral = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Neutral 0%", bg = "floral white")
        self.label_neutral.place(relx = 0.5, rely = 0.35, relwidth = 1.0, relheight = 0.15, anchor = 'center')
        self.label_disgust = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Disgust 0%", bg = "floral white")
        self.label_disgust.place(relx = 0.5, rely = 0.5, relwidth = 1.0, relheight = 0.15, anchor = 'center')
        self.label_sad = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Sad 0%", bg = "floral white")
        self.label_sad.place(relx = 0.5, rely = 0.65, relwidth = 1.0, relheight = 0.15, anchor = 'center')
        self.label_fearful = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Fearful 0%", bg = "floral white")
        self.label_fearful.place(relx = 0.5, rely = 0.8, relwidth = 1.0, relheight = 0.15, anchor = 'center')
        self.label_angry = tk.Label(self.frame_emotion_count, font = ("Verdana 12 bold"), justify = "left", text = "Angry 0%", bg = "floral white")
        self.label_angry.place(relx = 0.5, rely = 0.95, relwidth = 1.0, relheight = 0.15, anchor = 'center')

        ############################### BUTTONS ##############################
        #Create a button for starting microphone recording
        self.record_button = tk.Button(GUI, 
                                       text = "Start Recording",
                                       image = self.record_icon,
                                       bg = "light steel blue",
                                       borderwidth = 0,
                                       command = self.start_recording
                                      )
        self.record_button.img = self.record_icon
        self.record_button.place(relx = 0.5, rely = 0.9, relwidth = 0.15, relheight = 0.15, anchor = 'center')

        #Create a button for stopping microphone recording
        self.stop_record_button = tk.Button(GUI,
                                            text = "Stop Recording",
                                            image = self.stop_icon,
                                            bg = "light steel blue",
                                            borderwidth = 0,
                                            command = self.stop_recording
                                           )
        #self.stop_record_button.place(relx = 0.65, rely = 0.9, relwidth = 0.2, relheight = 0.1, anchor = 'center')

        #Create a button for importing audio file
        self.import_button = tk.Button(GUI, 
                                       text = "Import Audio File", 
                                       image = self.import_icon,
                                       bg = "light steel blue",
                                       borderwidth = 0,
                                       command = self.start_audio_file_transcription
                                      )
        #self.import_button.place(relx = 0.4, rely = 0.9, relwidth = 0.2, relheight = 0.1, anchor = 'center')
        
    #################################### BUtton commands ##########################################################
    def start_recording(self):
        if not self.is_recording.get(): #Prevents transcription process from occuring repeatedly
            self.is_recording.set(True)

            #Clear the text window
            self.transcription_window.config(state = "normal")
            self.transcription_window.delete('1.0', tk.END)
            self.transcription_window.config(state = "disabled")
            reset_emotion_percentage(self)
            #Run the live transcription and emotion detection in a separate thread to prevent GUI from lagging
            thread = threading.Thread(target = perform_live_recording, args = (self,))
            thread.daemon = True
            #thread.start()
            GUI.after(500, lambda: thread.start())

            #Hide the start recording button and show the stop recording button
            self.record_button.place_forget()
            self.stop_record_button.place(relx = 0.5, rely = 0.9, relwidth = 0.15, relheight = 0.15, anchor = 'center')
            #Disable the radiobutton settings while recording to prevent unwanted errors
            self.option_microphone_radiobutton.configure(state = "disabled")
            self.option_import_radiobutton.configure(state = "disabled")

    def stop_recording(self):
        self.is_recording.set(False)
        #Hide the stop recording button and show the start recording button
        self.stop_record_button.place_forget()
        self.record_button.place(relx = 0.5, rely = 0.9, relwidth = 0.15, relheight = 0.15, anchor = 'center')

        #Re-enable the radiobutton settings
        self.option_microphone_radiobutton.configure(state = "normal")
        self.option_import_radiobutton.configure(state = "normal")
        print("Stopped Recording")

    def start_audio_file_transcription(self):
        #Run the transcription and emotion detection in a separate thread to prevent GUI from lagging
        thread = threading.Thread(target = import_audio_file, args = (self,))
        thread.daemon = True
        GUI.after(500, lambda: thread.start())

    def show_microphone_input_buttons(self):
        #Hide the import button
        self.import_button.place_forget()
        #Show the microphone input button functions
        self.record_button.place(relx = 0.5, rely = 0.9, relwidth = 0.15, relheight = 0.15, anchor = 'center')

    def show_import_audio_file(self):
        #Hide the microphone input button functions
        self.record_button.place_forget()
        self.stop_record_button.place_forget()
        #Show the import audio button
        self.import_button.place(relx = 0.5, rely = 0.9, relwidth = 0.15, relheight = 0.15, anchor = 'center')




#GUI.after(1, ProduceTextOutput)
GUI = tk.Tk()
GUI.title("Speech Emotion Recognition App")
#Initialize the speech-to-text model
whisper_model, classes = perform_initialization()
GUI.configure(background = "light steel blue")
GUI.geometry("1200x1000")
Components = GuiComponents(GUI, whisper_model, classes)
GUI.protocol("WM_DELETE_WINDOW", lambda:close_application(Components.is_recording))
GUI.mainloop()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 